# AuditDDI: Reproducible Colab Training and Research Pipeline

This notebook runs the **AuditDDI** pharmacoinformatics training pipeline on Google Colab GPU runtimes.

### Active Datasets (9 Total):
The project utilizes the following 9 curated biological and chemical datasets located in `auditddi-data`:
1. **TWOSIDES**: Polypharmacy drug-drug interactions with 1,317 adverse effect types.
2. **FAERS**: Real-world FDA Adverse Event Reporting System post-marketing clinical reports.
3. **UniProt**: Authoritative Swiss-Prot/UniProtKB target protein sequences and metadata.
4. **BindingDB**: Experimentally measured drug-target binding affinities (Ki, Kd, IC50).
5. **PharmGKB**: Curated pharmacogenomics relationships, variant annotations, and biological pathways.
6. **ChEMBL**: 2.4M+ bioactive chemical structures and target mapping for self-supervised pretraining.
7. **PubChem**: Standardized chemical synonym dictionaries, biologics, and CID catalogs.
8. **PDB**: Experimental 3D macromolecular structures for structural binding verification.
9. **GEO**: Functional transcriptomics expression profiles (Cardiovascular Heart Failure & Alzheimer's).

### Google Drive Shared Setup:
- **`auditddi-data`**: Added directly to `MyDrive` across all rotating Google Colab accounts with full edit permissions.
- Output results and checkpoints can be written directly to `/content/drive/MyDrive/auditddi-data/results` or `/content/drive/MyDrive/auditddi-results`.


## Step 1: Connect Google Drive and Clone Repository


In [ ]:
from google.colab import drive
import os
import sys

# Mount Google Drive
drive.mount('/content/drive')

# Check if codebase exists in Drive or clone from GitHub
drive_code_path = '/content/drive/MyDrive/AuditDDI'
if os.path.exists(drive_code_path) and os.path.exists(os.path.join(drive_code_path, 'src')):
    print('[OK] Using AuditDDI codebase directly from Google Drive:', drive_code_path)
    %cd /content/drive/MyDrive/AuditDDI
else:
    print('[OK] Cloning/updating latest AuditDDI from GitHub...')
    if not os.path.exists('/content/auditddi'):
        !git clone https://github.com/akhilau-git/auditddi.git /content/auditddi
    %cd /content/auditddi
    !git pull origin main

# Add current directory to Python module search path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('[OK] Working directory ready:', os.getcwd())


## Step 2: Install Colab Dependencies (GPU)


In [ ]:
!pip install -r requirements_colab.txt


## Step 3: Audit and Verify All 9 Dataset Folders in Google Drive
Runs an exhaustive census checking all 9 active folders, computing exact sizes, file counts, and verifying critical files.

In [ ]:
!python scripts/colab_drive_census.py


## Step 3b (Optional): Download UniProt Targets if Missing
If the census indicates `UniProt` is not yet populated in Drive, run this cell to fetch real-time Swiss-Prot target sequences:

In [ ]:
# !python -m src.data_prep.download_uniprot_data --output-dir /content/drive/MyDrive/auditddi-data/UniProt


## Step 3c: Extract FAERS Disproportionality Safety Signals (ROR / PRR)
Processes FDA adverse-event reports in `auditddi-data/FAERS` to compute Reporting Odds Ratios (ROR) and 95% confidence intervals for post-marketing safety.

In [ ]:
!python -m src.data_prep.faers_pipeline --min-reports 5


## Step 3d: Build Unified Multimodal Knowledge Graph Cache
Fuses TWOSIDES interactions, PharmGKB enzymes, FAERS safety signals, and BindingDB affinities into the master multimodal graph.

In [ ]:
!python scripts/build_full_knowledge_graph.py


## Step 4: Run GNN Training (Edge-Aware GATv2 Candidate)


In [ ]:
import os

# Configure AuditDDI Environment Variables
os.environ['AUDITDDI_DATA_BASE'] = '/content/drive/MyDrive/auditddi-data'
os.environ['AUDITDDI_RESULTS_BASE'] = '/content/drive/MyDrive/auditddi-results'
os.environ['AUDITDDI_MODEL_ARCHITECTURE'] = 'edge_aware_gat_v2'
os.environ['AUDITDDI_NEGATIVE_SAMPLING_PROTOCOL'] = 'split_aware_standard_v1'
os.environ['AUDITDDI_NEGATIVE_SAMPLING_STRATEGY'] = 'degree_matched'
os.environ['AUDITDDI_EPOCHS'] = '200'
os.environ['AUDITDDI_BATCH_SIZE'] = '128'

!python -m src.training.train_full_pipeline_v2


## Step 5: Run Cold-Target Benchmark Evaluation (Multimodal)


In [ ]:
!python -m src.training.benchmark_cold_target \
    --master_nodes /content/drive/MyDrive/auditddi-results/unified_graph/master_drug_nodes_verified_targets.csv \
    --splits_dir /content/drive/MyDrive/auditddi-results/benchmark_splits \
    --output_dir /content/drive/MyDrive/auditddi-results/benchmark_cold_target_results \
    --epochs 10 \
    --batch_size 128 \
    --lr 0.0002 \
    --cold_sim_dropout 0.30 \
    --force_retrain

from pathlib import Path
from IPython.display import display, Markdown
report_path = Path('/content/drive/MyDrive/auditddi-results/benchmark_cold_target_results/cold_target_performance_comparison.md')
if report_path.is_file():
    print('\n' + '=' * 80)
    print('  AUDITDDI MULTIMODAL BENCHMARK EVALUATION REPORT')
    print('=' * 80)
    display(Markdown(report_path.read_text(encoding='utf-8')))


## Step 6: Biophysical ADMET, Guardrail Abstention & Multi-Drug Polypharmacy Audit
Runs the first-principles chemistry, biology, and physics reasoning engine to assess:
1. Single-drug intrinsic toxicity and ADMET.
2. Multi-stage guardrail check (abstains on out-of-distribution molecules).
3. Research-only mechanistic risk signals and severe metabolic-bottleneck warnings; no clinical safety determination.
4. Multi-drug polypharmacy cumulative CYP clearance load across multiple drugs.


In [ ]:
from src.evaluation.clinical_audit_report import generate_clinical_audit_report, generate_polypharmacy_audit_report
from IPython.display import Markdown, display

print('=== 1. RESEARCH AUDIT: LOWER-SIGNAL EXAMPLE PAIR ===')
# Paracetamol + Amoxicillin
safe_report = generate_clinical_audit_report('CC(=O)Nc1ccc(O)cc1', 'CC1(C)SC2C(NC(=O)C(N)c3ccc(O)cc3)C(=O)N2C1C(=O)O')
display(Markdown(safe_report['markdown']))

print('\n=== 2. CLINICAL AUDIT: HIGH-RISK NTI COLLISION ===')
# Warfarin + Fluconazole
high_risk = generate_clinical_audit_report('CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O', 'OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F')
display(Markdown(high_risk['markdown']))

print('\n=== 3. MULTI-DRUG POLYPHARMACY REGIMEN AUDIT ===')
# Warfarin + Fluconazole + Procaine
poly = generate_polypharmacy_audit_report([
    'CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O',
    'OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F',
    'CCN(CC)CCOC(=O)c1ccc(N)cc1'
], drug_names=['Warfarin', 'Fluconazole', 'Procaine'])
display(Markdown(poly['markdown']))
